# Week 1 lab — can one arrival rate describe a call centre?

**Getting the course materials.** We recommend forking the course repository on GitHub and working in your local clone. A fork lets you receive later course updates without putting your work in the course repository. After creating your fork, run:

```bash
git clone https://github.com/<your-account>/stochastic-modeling-simulation-ds.git
cd stochastic-modeling-simulation-ds
git remote add upstream https://github.com/NaitsabesMue/stochastic-modeling-simulation-ds.git
git pull --ff-only upstream main
```

Repeat `git pull --ff-only upstream main` when the lecturer announces an update. After pulling, make a working copy of this notebook and open the copy:

```bash
mkdir -p students/week01
cp notebooks/lecture1_poisson_warmup.ipynb students/week01/lecture1_poisson_warmup.ipynb
```

Work only in `students/week01/lecture1_poisson_warmup.ipynb`; leave the tracked notebook in `notebooks/` unchanged. If you pull a later course update, make a fresh copy only after saving your current work. The entire `students/` folder is ignored: never commit or push student work. VS Code with its Python/Jupyter extensions is convenient, but JupyterLab, Spyder, PyCharm, or another Python IDE is equally acceptable.

**Create a local Python environment.** From the repository folder, create one environment for this course and activate it. On macOS/Linux:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install jupyter ipykernel
```

On Windows PowerShell, use `py -m venv .venv`, then `.venv\Scripts\Activate.ps1` and the same two `python -m pip ...` commands. In VS Code or another IDE, select the interpreter inside `.venv`. The notebook itself uses only Python's standard library; `jupyter` and `ipykernel` are needed only to run it as a notebook. When finished, leave the environment with `deactivate`. The `.venv/` directory is local and ignored by Git.

Once you are ready let us start with the first question:

**Question.** Does one constant arrival rate look credible throughout a day? Start with the 96 counts from 1 November, then test a baseline that uses the same rate for every interval in the month. We will observe real aggregate counts, build a simple stochastic model, simulate it in plain Python, and check where it fails. The last step is to say what these counts cannot tell us about a staffing change.

**Data.** The course extract comes from the [Technion SEE Lab *AnonymousBank* data](https://seelab.net.technion.ac.il/data/), November 1999. Each row gives the number of non-PHANTOM calls entering the voice response unit (VRU) during one 15-minute interval. It contains no call, customer, or agent identifiers. Counts alone do not tell us when service began, how long it lasted, or how long anyone waited. Read the [dataset guide](../data/anonymousbank_data_guide.md) for the call path, outcome labels, raw fields, and limitations.

**Session routine.** The opening lecture and diagnostic happen on paper with student screens closed. During the lab you may use Python, the Internet and LLMs, and discuss with your group, but write your own code, answers and reasoning. Check any LLM output independently. Write short answers below each task or in new cells, and run the notebook from top to bottom before the exit quiz.

## 0. Warmup 

### 1. Python : conversion, intervalles vides, proportion

> Convert the counts to integers. Write an expression for the **number** of empty intervals, then calculate their **proportion**. Explain the denominator.

```python
counts = [int(s) for s in practice_text_counts]    # [0, 1, 3, 1, 0, 5]
n_empty = sum(1 for c in counts if c == 0)         # nombre d'intervalles vides
prop_empty = n_empty / len(counts)                 # proportion
```

**Réponse.** Il y a **2 intervalles vides**, donc une proportion de **2/6 ≈ 33 %**.
Le dénominateur (6) est le **nombre total d'intervalles observés**, vides ou non : une proportion se calcule sur l'ensemble des intervalles, pas seulement sur ceux où il se passe quelque chose.



### 2. Espérance vs comptage exact

> A model predicts 2 entries *on average* per interval. Does seeing 5 in one interval disprove the model? Explain the difference between an expected count and an exact count.

**Réponse.**
- **Non, voir 5 dans un intervalle ne réfute pas le modèle.** Pour une Poisson(2), P(X = 5) ≈ 3,6 % et P(X ≥ 5) ≈ 5,3 % : c'est peu fréquent mais tout à fait plausible.
- Une **espérance** de 2, c'est la moyenne sur un très grand nombre d'intervalles.
- Un **comptage exact**, c'est une réalisation aléatoire : il fluctue autour de l'espérance.



### 3. Contrôle du modèle : moyenne égale, variances différentes

> The two integer lists both have mean 2. Compute their population variances. Why is matching the mean alone insufficient when checking a demand model?

**Réponse.** `statistics.pvariance` donne **0** pour `[2, 2, 2, 2]` et **4** pour `[0, 0, 4, 4]`, alors que les deux listes ont une moyenne de 2.

| Liste | Moyenne | Variance | Par rapport à Poisson(2) (variance 2) |
|---|---|---|---|
| A = `[2, 2, 2, 2]` | 2 | 0 | trop régulière |
| B = `[0, 0, 4, 4]` | 2 | 4 | trop dispersée |

La moyenne seule ne suffit pas, car un modèle de demande fait des prédictions sur la **variance**, la **part de zéros**, l'**indépendance**… et pas seulement sur le niveau moyen.



### 4. Décision : trois mesures manquantes

> These counts record arrivals only. To estimate how one extra agent changes waiting, name three missing measurements and explain why each matters.

**Réponse.**
1. **Les durées de service** : elles déterminent la charge de travail par appel (arrivées × durée de service).
2. **Le nombre d'agents présents au fil du temps** : c'est la capacité du système, nécessaire pour savoir ce que change « un agent de plus ».
3. **Les heures d'entrée et de sortie de file, et les abandons** : elles permettent de mesurer l'attente, qui est la quantité qu'on cherche à réduire, et de valider le modèle.


In [24]:
practice_text_counts = ["0", "1", "3", "1", "0", "5"]
equal_mean_a = [2, 2, 2, 2]
equal_mean_b = [0, 0, 4, 4]
print("CSV-like counts:", practice_text_counts)
print("Equal-mean lists:", equal_mean_a, equal_mean_b)


CSV-like counts: ['0', '1', '3', '1', '0', '5']
Equal-mean lists: [2, 2, 2, 2] [0, 0, 4, 4]


In [25]:
counts = [int(s) for s in practice_text_counts]
n_empty = sum(1 for c in counts if c == 0)      # ou counts.count(0)
prop_empty = n_empty / len(counts)
print("Number of empty counts:", n_empty)
print("Proportion of empty counts:", prop_empty)

Number of empty counts: 2
Proportion of empty counts: 0.3333333333333333


In [26]:
# Q2 : combien est-il "rare" de voir 5 quand la moyenne vaut 2 ?
def poisson_pmf(k, mu):
    return math.exp(-mu) * mu**k / math.factorial(k)

print("P(X=5)  =", round(poisson_pmf(5, 2), 4))
print("P(X>=5) =", round(1 - sum(poisson_pmf(k, 2) for k in range(5)), 4))

# Q3
print("moyennes :", statistics.mean(equal_mean_a), statistics.mean(equal_mean_b))
print("variances :", statistics.pvariance(equal_mean_a), statistics.pvariance(equal_mean_b))

P(X=5)  = 0.0361
P(X>=5) = 0.0527
moyennes : 2 2
variances : 0 4


## 1. Observe: load the real counts

The CSV has three columns: `date` (local calendar day), `time` (start of a 15-minute interval), and `entries` (arrival count). The loader works when you launch Jupyter from the repository root or from `notebooks/`.

**Before running:** How many rows should November have if every day has 96 quarter-hour intervals?

$$30 \times 24 \times 4 = 30 \times 96 = 2\,880 \text{ lignes}$$


In [27]:
import csv
import math
import random
import statistics
from pathlib import Path

filename = "anonymousbank_nov1999_arrivals_15min.csv"
candidates = [Path.cwd() / "data" / filename, Path.cwd().parent / "data" / filename]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        f"Cannot find {filename} from {Path.cwd()}. Start Jupyter in the repository root or notebooks/."
    )

with data_path.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    if reader.fieldnames != ["date", "time", "entries"]:
        raise ValueError(f"Unexpected CSV columns: {reader.fieldnames}")
    rows = [
        {"date": row["date"], "time": row["time"], "entries": int(row["entries"])}
        for row in reader
    ]

print(f"Loaded {len(rows)} intervals from {data_path}")
print("First eight intervals:")
for row in rows[:8]:
    print(f"{row['date']}  {row['time']}  {row['entries']:3d}")


Loaded 2880 intervals from g:\Mon Drive\Documents GD\M2\RENF INFO 1 MULLER\stochastic-modeling-simulation-ds\data\anonymousbank_nov1999_arrivals_15min.csv
First eight intervals:
1999-11-01  00:00    5
1999-11-01  00:15    2
1999-11-01  00:30    0
1999-11-01  00:45    1
1999-11-01  01:00    0
1999-11-01  01:15    3
1999-11-01  01:30    0
1999-11-01  01:45    0


Check the extract before fitting anything. A missing or duplicated interval would distort both the estimated rate and the time-of-day comparison. The assertions below are tailored to this teaching file: November 1999, 30 days, 96 quarter-hour rows per day, and 40,722 total entries. If you load another month or dataset, change the expected dates, slots and total accordingly.


In [28]:
dates = sorted({row["date"] for row in rows})
slots_by_date = {
    day: {row["time"] for row in rows if row["date"] == day}
    for day in dates
}
expected_dates = [f"1999-11-{day:02d}" for day in range(1, 31)]
expected_slots = {f"{hour:02d}:{minute:02d}" for hour in range(24) for minute in (0, 15, 30, 45)}
assert dates == expected_dates, "Expected every date in November 1999"
assert all(slots == expected_slots for slots in slots_by_date.values())
assert len(rows) == 30 * 96, "Expected 30 complete November days"
assert len(dates) == 30 and all(len(slots) == 96 for slots in slots_by_date.values())
assert len({(row["date"], row["time"]) for row in rows}) == len(rows)
assert rows == sorted(rows, key=lambda row: (row["date"], row["time"]))
assert all(row["entries"] >= 0 for row in rows)

observed = [row["entries"] for row in rows]
assert sum(observed) == 40722, "Total differs from the documented extract"
print(f"Dates: {dates[0]} to {dates[-1]}; total VRU entries: {sum(observed):,}")


Dates: 1999-11-01 to 1999-11-30; total VRU entries: 40,722


### First look: the 96 intervals on 1 November

This is the same day as the lecture plot. Each line below is one CSV row; each `#` represents one VRU entry in that 15-minute interval. There is no smoothing or averaging across intervals or days. These are aggregated call counts, not individual call timestamps.

Does a constant rate look credible for this day? Identify a quiet period and a busy period. Counts fluctuate even at a fixed rate, so a changing count alone does not establish a changing rate. To investigate whether the daily pattern also changes by weekday, we must compare multiple days.




**Non, un taux constant n'est pas crédible sur cette journée.**

**Période calme.** De **01:30 à 06:00**, 19 intervalles de suite avec 0 ou 1 entrée : presque tous sont à 0, avec seulement trois fois 1 entrée (04:15, 05:00 et 05:45).

**Période chargée.** La matinée, de **09:00 à 10:45**, avec 40 à 56 entrées par intervalle. Le maximum de la journée est de **56 entrées à 09:15**.

**Pourquoi un taux constant ne tient pas.**
- Sur la journée, la moyenne est de 20,9 entrées par intervalle. Pour une loi de Poisson de moyenne 20,9, l'écart-type est $\sqrt{20{,}9} \approx 4{,}6$, donc presque tous les comptages devraient être entre 7 et 35 (moyenne ± 3 écarts-types).
- Or **50 intervalles sur 96** sont en dehors de cette bande (des 0 la nuit, un pic à 56 le matin), alors qu'on en attendrait environ 0 ou 1 sous le modèle.
- On observe **18 intervalles vides**, alors que la probabilité d'un intervalle vide sous Poisson(20,9) est $e^{-20{,}9} \approx 8 \times 10^{-10}$.
- J'ai centré la bande sur la meilleure moyenne possible pour ce jour : si même ce taux constant « le plus favorable » ne convient pas, aucun autre taux constant ne conviendra.

**Ce que ça ne prouve pas.** Un comptage qui fluctue ne suffit pas à prouver que le taux change, car une Poisson fluctue déjà à taux fixe. Ce qui est convaincant ici, c'est l'**ampleur** de l'écart, et pas le simple fait que les chiffres varient.

**Limite : le jour de la semaine.** Le 1er novembre 1999 est un lundi, et une seule journée ne permet pas de dire si le profil est le même les autres jours. Pour savoir si le motif change selon le jour, il faut comparer plusieurs dates, en gardant en tête que les jours ouvrés documentés sont du dimanche au jeudi.

In [29]:
first_day_rows = [row for row in rows if row["date"] == "1999-11-01"]
assert len(first_day_rows) == 96
print("1 November 1999: observed counts, one # per entry")
print("Time   Entries")
for row in first_day_rows:
    print(f"{row['time']}  {row['entries']:3d}  " + "#" * row["entries"])


1 November 1999: observed counts, one # per entry
Time   Entries
00:00    5  #####
00:15    2  ##
00:30    0  
00:45    1  #
01:00    0  
01:15    3  ###
01:30    0  
01:45    0  
02:00    0  
02:15    0  
02:30    0  
02:45    0  
03:00    0  
03:15    0  
03:30    0  
03:45    0  
04:00    0  
04:15    1  #
04:30    0  
04:45    0  
05:00    1  #
05:15    0  
05:30    0  
05:45    1  #
06:00    0  
06:15    3  ###
06:30    4  ####
06:45    5  #####
07:00   14  ##############
07:15   11  ###########
07:30   13  #############
07:45   19  ###################
08:00   18  ##################
08:15   35  ###################################
08:30   25  #########################
08:45   39  #######################################
09:00   40  ########################################
09:15   56  ########################################################
09:30   49  #################################################
09:45   51  ###################################################
10:00   49  #######

In [30]:
import math, statistics



# 2) Repérer une période calme et une période chargée
counts = [r["entries"] for r in day]
busiest = max(day, key=lambda r: r["entries"])
print("\nIntervalle le plus chargé :", busiest["time"], "->", busiest["entries"])

quiet, current = [], []
for r in day:
    if r["entries"] <= 1:
        current.append(r["time"])
    else:
        quiet = max(quiet, current, key=len)
        current = []
quiet = max(quiet, current, key=len)
print("Plus longue période calme (<= 1 entrée) :", quiet[0], "à", quiet[-1], f"({len(quiet)} intervalles)")

# 3) Test rapide : un taux constant est-il crédible sur cette journée ?
mu = statistics.mean(counts)          # taux constant "ajusté" sur la journée
sd = math.sqrt(mu)                    # écart-type d'une Poisson(mu)
low, high = mu - 3 * sd, mu + 3 * sd
outside = [r for r in day if not low <= r["entries"] <= high]

print(f"\nMoyenne du jour : {mu:.1f} par intervalle, écart-type Poisson : {sd:.1f}")
print(f"Intervalles hors de [{low:.0f} ; {high:.0f}] : {len(outside)} sur 96")
print(f"P(intervalle vide | Poisson({mu:.1f})) = {math.exp(-mu):.1e}, "
      f"intervalles vides observés : {counts.count(0)}")


Intervalle le plus chargé : 09:15 -> 56
Plus longue période calme (<= 1 entrée) : 01:30 à 06:00 (19 intervalles)

Moyenne du jour : 20.9 par intervalle, écart-type Poisson : 4.6
Intervalles hors de [7 ; 35] : 50 sur 96
P(intervalle vide | Poisson(20.9)) = 8.0e-10, intervalles vides observés : 18


### Summarise all November intervals

Before running the next cell, predict whether a randomly chosen 15-minute interval from the **whole month** is likely to contain no entries. Then compare your prediction with the data.

The variance below uses divisor $n$, so it describes the observed set of bins. The ratio variance/mean is often called a *dispersion index*.


**Ma prédiction (écrite avant d'exécuter).**
Avec une moyenne de l'ordre de 14 entrées par intervalle, je m'attends à ce que très peu d'intervalles du mois soient vides.

**Ce que calcule la cellule.**
- `observed_mean` : la moyenne des 2 880 comptages.
- `observed_variance` : la variance avec diviseur $n$, car on décrit **ces** 2 880 intervalles observés et non un échantillon tiré d'une population plus grande.
- `variance / mean` : l'**indice de dispersion**. Pour une loi de Poisson, variance et moyenne sont égales, donc cet indice vaut 1.
- `observed_zero_share` : la part d'intervalles vides, c'est-à-dire le nombre d'intervalles à 0 divisé par 2 880.


In [31]:
observed_mean = statistics.mean(observed)
observed_variance = statistics.pvariance(observed)
observed_zero_share = observed.count(0) / len(observed)

print(f"Mean entries per 15 minutes: {observed_mean:.2f}")
print(f"Variance across 15-minute bins: {observed_variance:.2f}")
print(f"Variance / mean: {observed_variance / observed_mean:.2f}")
print(f"Share of empty bins: {observed_zero_share:.1%}")
print(f"Largest count in one bin: {max(observed)}")


Mean entries per 15 minutes: 14.14
Variance across 15-minute bins: 209.30
Variance / mean: 14.80
Share of empty bins: 29.1%
Largest count in one bin: 78


**Dispersion.** L'indice variance/moyenne est d'environ 14.8, très loin de 1. Les comptages sont donc beaucoup plus dispersés que ne le prévoirait une loi de Poisson à taux constant. C'est cohérent avec ce qu'on a vu sur le 1er novembre : des nuits quasi vides et des matinées très chargées.

**Attention.** Ces résultats décrivent des entrées dans le VRU, pas des arrivées dans la file d'attente des agents.

**Task 1.** What do the mean and empty-bin share say about a typical interval? Why might a whole-month average hide something important? Give two reasons, and label each one:

- **System or operational reason:** something about how the call centre actually behaves, such as quiet nights and busy mornings needing different staffing.
- **Data or measurement reason:** something about how the records were collected or summarised, such as 15-minute counts hiding the exact arrival times or VRU entries not being queue arrivals.

For each reason, write one sentence explaining how it could make the month-wide average misleading.


### Task 1 : que disent la moyenne et la part d'intervalles vides ?

**Intervalle typique.**
La moyenne est d'environ 14.1 entrées par intervalle de 15 min, mais **29.1 % des intervalles sont vides** (837 sur 2 880) et le maximum est de 78. Aucun intervalle n'est vraiment « typique » : la moyenne de 14 ne décrit ni les intervalles vides ni les plus chargés. Si les comptages suivaient une loi de Poisson de moyenne 14.1, un intervalle vide serait quasi impossible, donc ces chiffres montrent que le mois mélange des périodes très différentes.

**Raison système / opérationnelle.**
Le centre d'appels n'a pas la même activité selon l'heure et le jour : environ 0.1 à 1.3 entrée par intervalle entre 00h et 06h, contre plus de 30 entre 09h et 11h, et les vendredis et samedis sont bien plus calmes que les jours ouvrés du dimanche au jeudi. Une moyenne unique de 14 sous-estime donc la demande aux heures de pointe et la surestime la nuit, alors que ce sont justement ces différences qui déterminent le nombre d'agents nécessaire à chaque moment.

**Raison données / mesure.**
Les comptages sont agrégés par intervalles de 15 minutes et correspondent à des entrées dans le VRU, pas à des arrivées dans la file d'attente des agents : la moyenne de 14 entrées ne dit donc ni quand exactement les appels arrivent, ni combien de ces appels demandent réellement un agent, ce qui la rend trompeuse pour estimer la charge et l'attente.

## 2. Model: a homogeneous Poisson arrival process

A first model assumes a **constant** arrival rate $r$ per minute. Under that model, arrivals in disjoint time intervals are independent; each 15-minute count has a Poisson distribution with mean $15r$. Its theoretical variance is also $15r$, and its probability of zero arrivals is $e^{-15r}$. Those are model claims to check, not properties guaranteed by the log.

Estimate $r$ from the whole-month average count. This deliberately simple estimate uses every 15-minute interval, including nights, busy hours and different days of the week. This is a deliberate baseline: one rate for all 2,880 intervals, rather than a rate fitted only to the day shown above. Keep track of units.


In [32]:
bin_minutes = 15
rate_per_minute = observed_mean / bin_minutes
model_zero_probability = math.exp(-observed_mean)

print(f"Estimated rate: {rate_per_minute:.3f} entries per minute")
print(f"Model probability of an empty 15-minute bin: {model_zero_probability:.4%}")
print(f"Observed share of empty bins: {observed_zero_share:.1%}")


Estimated rate: 0.943 entries per minute
Model probability of an empty 15-minute bin: 0.0001%
Observed share of empty bins: 29.1%


**Task 2.** Is the difference in empty-bin shares plausibly explained by sampling variation under the model? What does this comparison say about the *constant-rate assumption*?


In [33]:
expected_empty = len(observed) * model_zero_probability
print(f"Intervalles vides attendus sous le modèle : {expected_empty:.4f}")
print(f"Intervalles vides observés : {observed.count(0)}")

Intervalles vides attendus sous le modèle : 0.0021
Intervalles vides observés : 837


### Task 2 : l'écart sur les intervalles vides s'explique-t-il par la variation d'échantillonnage ?

**Réponse : non.**

**Ce que prédit le modèle.** Avec un taux constant estimé à partir de la moyenne ($\bar{x} \approx 14.14$ entrées par intervalle), la probabilité d'un intervalle vide est

$$P(\text{vide}) = e^{-14.14} \approx 7.2 \times 10^{-7}$$

soit environ 0.00007 %.

**Nombre d'intervalles vides attendu sur le mois.**

$$2880 \times 7.2 \times 10^{-7} \approx 0.002$$

Sous le modèle, on s'attend donc à n'observer **aucun** intervalle vide : la probabilité d'en voir ne serait-ce qu'un seul est d'environ 0.2 %.

**Ce qu'on observe.** 837 intervalles vides sur 2 880, soit 29.1 %.

**Pourquoi la variation d'échantillonnage ne l'explique pas.** Passer d'environ 0.002 intervalle vide attendu à 837 observés n'est pas une fluctuation aléatoire : c'est un écart de plusieurs ordres de grandeur. Un autre échantillon tiré du même modèle donnerait de nouveau zéro intervalle vide, ou un seul au maximum.

**Ce que ça dit sur l'hypothèse de taux constant.** Elle est rejetée. Le taux réel n'est pas le même à toutes les heures : il est très inférieur à la moyenne du mois pendant certaines périodes (surtout la nuit), ce qui produit beaucoup d'intervalles vides. Ce résultat va dans le même sens que l'indice de dispersion d'environ 14.8, très loin de la valeur 1 attendue pour une loi de Poisson.

**Limite.** Cette comparaison teste le niveau et la part de zéros, pas l'indépendance entre intervalles voisins : on peut conclure que *le taux constant* est rejeté, mais pas que toutes les hypothèses du modèle sont fausses pour toutes leurs raisons.

### Simulate arrival times with plain Python

For a constant-rate Poisson process, successive times between arrivals are independent exponential random variables with mean $1/r$ minutes. Add each gap to a clock, then place the arrival in its 15-minute bin. We simulate as many bins as the real November extract.

Read the function before running it. In particular, explain why `int(clock // minutes_per_bin)` identifies the bin and why the loop stops at `horizon`.


In [34]:
def simulate_arrival_counts(n_bins, minutes_per_bin, rate_per_minute, rng):
    """Simulate constant-rate arrivals and return counts in consecutive bins."""
    if n_bins <= 0 or minutes_per_bin <= 0 or rate_per_minute <= 0:
        raise ValueError("Number of bins, bin width, and rate must be positive")
    counts = [0] * n_bins
    horizon = n_bins * minutes_per_bin
    clock = rng.expovariate(rate_per_minute)
    while clock < horizon:
        counts[int(clock // minutes_per_bin)] += 1
        clock += rng.expovariate(rate_per_minute)
    return counts

simulated = simulate_arrival_counts(
    len(observed), bin_minutes, rate_per_minute, random.Random(2026)
)
print(f"Simulated {sum(simulated):,} arrivals in {len(simulated)} bins")


Simulated 41,048 arrivals in 2880 bins


### Check the simulator; then check the model

First compare simulated statistics with the model's predictions. The mean and variance should both be close to $15r$; agreement supports the implementation but is not a complete proof of correctness. Then compare with the observed counts to assess the model.

The table below reproduces the lecture's run: 2,880 intervals, seed `2026`, simulated mean 14.25, variance 14.52 and no empty intervals. Both variances use divisor $n$. The theoretical empty-interval probability is only about 0.000072%, so zero empty intervals in this run is unsurprising. Other seeds give different results.


In [35]:
simulated_mean = statistics.mean(simulated)
simulated_variance = statistics.pvariance(simulated)
simulated_zero_share = simulated.count(0) / len(simulated)

print(f"{'Quantity':<23} {'Model':>12} {'Simulated':>12} {'Observed':>12}")
print(f"{'Mean count':<23} {observed_mean:12.2f} {simulated_mean:12.2f} {observed_mean:12.2f}")
print(f"{'Variance of counts':<23} {observed_mean:12.2f} {simulated_variance:12.2f} {observed_variance:12.2f}")
print(f"{'Empty intervals':<23} {model_zero_probability:12.6%} {simulated_zero_share:12.1%} {observed_zero_share:12.1%}")
print(f"Simulated variance / mean: {simulated_variance / simulated_mean:.2f}")


Quantity                       Model    Simulated     Observed
Mean count                     14.14        14.25        14.14
Variance of counts             14.14        14.52       209.30
Empty intervals            0.000072%         0.0%        29.1%
Simulated variance / mean: 1.02


**Task 3.** Run the simulator with two other seeds. Which numbers change? Which conclusion does not? Do not try to tune the seed until the simulation resembles the data.


In [36]:
for seed in (7, 11):
    trial = simulate_arrival_counts(len(observed), bin_minutes, rate_per_minute, random.Random(seed))
    trial_mean = statistics.mean(trial)
    print(
        f"seed {seed:>2}: mean {trial_mean:5.2f}, "
        f"variance/mean {statistics.pvariance(trial) / trial_mean:5.2f}, "
        f"empty bins {trial.count(0) / len(trial):5.1%}"
    )


seed  7: mean 14.17, variance/mean  0.99, empty bins  0.0%
seed 11: mean 14.17, variance/mean  1.02, empty bins  0.0%


### Conclusion de la partie 2

**1. Le simulateur est cohérent avec la théorie.**
Avec la graine 2026, la moyenne simulée est d'environ 14.25 et la variance d'environ 14.52, pour une valeur théorique de $15r \approx 14.14$ pour les deux. Moyenne et variance sont proches, comme le prévoit une loi de Poisson, et il n'y a aucun intervalle vide. Les graines 7 et 11 donnent le même constat (moyenne ≈ 14.17, rapport variance/moyenne entre 0.99 et 1.02, aucun intervalle vide). Ce qui change d'une graine à l'autre, ce sont ces petites valeurs. Ce qui ne change pas, c'est la conclusion. Je n'ai pas choisi de graine pour que la simulation ressemble aux données.

**2. Le modèle à taux constant est rejeté.**
Les données observées s'écartent fortement du modèle :

| Quantité | Modèle / simulation | Observé |
|---|---|---|
| Variance des comptages | ≈ 14.1 à 14.5 | ≈ 209.3 |
| Variance / moyenne | ≈ 1 | ≈ 14.8 |
| Part d'intervalles vides | ≈ 0 % | 29.1 % |

La variance observée est environ 15 fois trop grande pour une loi de Poisson, et on observe 837 intervalles vides alors que le modèle en prévoit environ 0.002. Aucune graine ne peut corriger un écart de cette taille : c'est le modèle qui est faux, pas le simulateur. Un simulateur correct peut parfaitement simuler un modèle faux.

**3. Ce qui manque au modèle.**
Le taux n'est probablement pas le même à toutes les heures ni tous les jours : le 1er novembre montrait des nuits presque vides et des matinées à plus de 40 entrées par intervalle. Mélanger tous ces intervalles dans un seul taux gonfle la variance et fait apparaître des vides. C'est une hypothèse que la section 3 va tester en comparant les moyennes heure par heure.

**4. Limite de cette partie.**
J'ai testé le niveau, la variance et la part d'intervalles vides. Je n'ai pas testé l'indépendance entre intervalles voisins. Je peux donc conclure que l'hypothèse de **taux constant** est rejetée, mais pas que toutes les hypothèses du modèle sont fausses.

## 3. Validate: when does the model fail?

A model can match the overall average and still fail at the times when decisions matter. Compare the average count for each hour of day with the simulator. Unlike the single-day display above, this table averages counts. For 10:00, for example, it pools the 10:00, 10:15, 10:30 and 10:45 counts from each of the 30 days: 120 counts in total. The result is an average number of entries per 15-minute interval, not an hourly total or a moving average. The bars show the observed mean; each `#` is roughly two arrivals per 15-minute bin.


In [37]:
observed_by_hour = [[] for _ in range(24)]
simulated_by_hour = [[] for _ in range(24)]
for row, simulated_count in zip(rows, simulated):
    hour = int(row["time"][:2])
    observed_by_hour[hour].append(row["entries"])
    simulated_by_hour[hour].append(simulated_count)

print(f"{'Hour':<6} {'Observed':>9} {'Simulated':>10}  Observed profile")
for hour in range(24):
    observed_hour_mean = statistics.mean(observed_by_hour[hour])
    simulated_hour_mean = statistics.mean(simulated_by_hour[hour])
    print(
        f"{hour:02d}:00  {observed_hour_mean:9.1f} {simulated_hour_mean:10.1f}  "
        + "#" * round(observed_hour_mean / 2)
    )


Hour    Observed  Simulated  Observed profile
00:00        1.1       14.0  #
01:00        0.4       13.9  
02:00        0.3       13.9  
03:00        0.1       14.1  
04:00        0.2       13.9  
05:00        0.3       15.1  
06:00        1.3       14.0  #
07:00        8.1       14.4  ####
08:00       17.3       14.2  #########
09:00       30.7       14.4  ###############
10:00       32.3       14.5  ################
11:00       27.0       14.0  ##############
12:00       23.7       14.2  ############
13:00       23.5       13.6  ############
14:00       25.2       13.8  #############
15:00       26.1       14.3  #############
16:00       20.7       14.7  ##########
17:00       16.0       14.3  ########
18:00       16.5       14.6  ########
19:00       16.4       13.6  ########
20:00       13.8       14.8  #######
21:00       14.7       15.2  #######
22:00       13.9       14.5  #######
23:00        9.8       14.3  #####


**Task 4.** Find an hour when the model greatly overpredicts entries and one when it underpredicts them. Suppose you used the month-wide rate to plan a morning shift: what might go wrong? Suggest one better arrival model that still uses only these counts. State one further check you would make before trusting it.

Time-of-day variation is visible here, but it need not be the only source of mismatch. Next compare the same clock times on different days to investigate a possible weekly pattern. One day alone cannot establish a repeating weekly pattern. The documented normal working days here are Sunday–Thursday, so do not automatically use a Monday–Friday split. Estimate any proposed rates on earlier days and check them on later days; special dates and the 15-minute aggregation also deserve attention. 


### Task 4 : où le modèle échoue-t-il, et que faire ?

**1. Une heure de forte surestimation et une de forte sous-estimation** (moyenne par intervalle de 15 min, 120 comptages par heure).

| Heure | Observé | Simulé (taux constant) | Écart |
|---|---|---|---|
| 03h00 | ≈ 0.1 | ≈ 14.1 | le modèle **surestime** d'environ 14 entrées |
| 10h00 | ≈ 32.3 | ≈ 14.5 | le modèle **sous-estime** d'environ 18 entrées |

La colonne simulée reste plate autour de 14, comme le prévoit un taux constant. La colonne observée varie, elle, de 0.1 à 32.3 selon l'heure.

**2. Que se passerait-il si on planifiait une équipe du matin avec le taux moyen ?**
Le taux moyen (≈ 14 entrées par intervalle) correspond à moins de la moitié de la demande réelle entre 09h et 11h (≈ 31 à 32 entrées par intervalle). L'équipe serait donc **sous-dimensionnée** aux heures de pointe : les appelants attendraient beaucoup plus longtemps ou raccrocheraient. À l'inverse, le même taux **surestime** la nuit, ce qui conduirait à payer des agents quasiment sans appels.

**3. Un meilleur modèle qui n'utilise que ces comptages.**
Un processus de Poisson dont le taux est **constant par morceaux** : un taux différent pour chaque heure (ou tranche horaire), estimé par la moyenne des comptages de cette heure, et éventuellement différent selon le type de jour. Le comptage d'un intervalle reste Poisson, mais sa moyenne dépend du moment. Comme les jours ouvrés documentés sont du **dimanche au jeudi**, le type de jour doit être défini ainsi (dimanche à jeudi contre vendredi et samedi), et non en lundi–vendredi.

**4. Une vérification avant de lui faire confiance.**
Estimer les taux sur les premiers jours du mois (par exemple les 20 premiers) et tester le modèle sur les jours suivants, qu'il n'a pas vus. Comparer l'erreur de prédiction du taux constant, du taux par heure et du taux par heure et type de jour : un modèle qui ne s'améliore que sur les données qui ont servi à l'estimer n'est pas fiable, car il colle mécaniquement à ses propres moyennes.

**Limites.** Même un taux par heure ne règle pas tout : il faut surveiller les dates spéciales, le fait que les comptages à 15 minutes cachent les instants exacts d'arrivée, et le fait que ces comptages sont des entrées dans le VRU, pas des arrivées dans la file d'attente.

In [38]:
import math
import statistics
from datetime import date

def is_working_day(day):
    return date.fromisoformat(day).weekday() in (6, 0, 1, 2, 3)   # dimanche à jeudi

days = sorted({r["date"] for r in rows})
train_days = set(days[:20])
train = [r for r in rows if r["date"] in train_days]
test  = [r for r in rows if r["date"] not in train_days]

def hourly_means(sample, split_by_daytype):
    groups = {}
    for r in sample:
        key = (int(r["time"][:2]), is_working_day(r["date"]) if split_by_daytype else None)
        groups.setdefault(key, []).append(r["entries"])
    return {k: statistics.mean(v) for k, v in groups.items()}

def rmse(predict):
    return math.sqrt(statistics.mean((r["entries"] - predict(r)) ** 2 for r in test))

const = statistics.mean(r["entries"] for r in train)
by_hour = hourly_means(train, False)
by_hour_type = hourly_means(train, True)

print(f"constant       : {rmse(lambda r: const):.2f}")
print(f"par heure      : {rmse(lambda r: by_hour[(int(r['time'][:2]), None)]):.2f}")
print(f"heure x type   : {rmse(lambda r: by_hour_type[(int(r['time'][:2]), is_working_day(r['date']))]):.2f}")

constant       : 13.73
par heure      : 8.60
heure x type   : 5.69


## 4. Decide: what is missing for a staffing question?

A manager asks: **“Will adding another agent reduce waiting times enough to justify the cost?”** The current extract contains arrival counts, but no service durations, agent schedules, queue entry/exit times, or abandonment records. It cannot answer that question on its own.

**Task 5.** List three additional measurements you would request. For each, say how it enters a queueing model or its validation. Then write a two-sentence recommendation to the manager: what the arrival analysis establishes, and what work is needed before making the staffing decision.


## 4. Décider : que manque-t-il pour une question de dimensionnement ?

### Task 5 : trois mesures supplémentaires

L'extrait ne contient que des **comptages d'entrées dans le VRU** par intervalle de 15 minutes : il décrit la demande, mais pas la façon dont elle se transforme en attente. Voici trois mesures que je demanderais.

**1. Les durées de service (`ser_time`).**
- *Rôle dans le modèle :* elles donnent la charge de travail. Avec un taux d'arrivée $\lambda$, une durée moyenne de service $E[S]$ et $c$ agents, le taux d'occupation vaut $\rho = \lambda \, E[S] / c$, et l'attente explose quand $\rho$ approche de 1.
- *Pourquoi c'est indispensable :* les mêmes arrivées peuvent représenter une charge faible ou saturée selon la durée des appels, donc on ne peut pas savoir si un agent de plus change quelque chose.
- *Précaution :* le guide des données signale que ces durées contiennent des services anormalement courts, qu'il faut nettoyer avant de les modéliser.

**2. Le nombre d'agents présents au fil du temps (planning).**
- *Rôle dans le modèle :* c'est la capacité $c(t)$, qui varie au cours de la journée.
- *Pourquoi c'est indispensable :* « un agent de plus » n'a de sens que par rapport à l'effectif actuel à chaque heure. La demande varie d'un facteur de l'ordre de 300 entre la nuit et le matin (partie 3), et sans le planning on ne sait pas si la capacité suit ce profil.

**3. Les heures d'entrée et de sortie de file, et les abandons (`q_time`, issues `HANG` avec un temps de file non nul).**
- *Rôle dans la validation :* elles donnent l'**attente observée**, que l'on compare à l'attente simulée par le modèle. Si les deux ne concordent pas, le modèle ne peut pas servir à prédire l'effet d'un agent de plus.
- *Rôle dans la calibration :* les abandons réduisent la charge réelle des agents. Un modèle qui les ignore surestime l'attente.
- *Remarque :* elles permettent aussi de séparer les entrées dans le VRU des vraies arrivées dans la file, qui sont moins nombreuses puisque certains appelants raccrochent sans jamais demander un agent.

### Recommandation au gestionnaire

L'analyse des arrivées établit que la demande varie fortement selon l'heure et le jour (nuits presque vides, matinées très chargées, vendredis et samedis bien plus calmes que les jours du dimanche au jeudi), donc un taux unique est inadapté et la capacité devrait suivre ce profil. Avant de décider d'ajouter un agent, il faut obtenir les durées de service, les plannings des agents et les heures d'entrée et de sortie de file avec les abandons, afin de construire un modèle de file d'attente et de le valider contre l'attente réellement observée.

## 5. Use an LLM as a theory checker

### Reflect on the delegation
Before opening an LLM, write down the task you will delegate. Afterward, record: (1) the exact claim or code suggestion, (2) what was useful, (3) what you checked independently, (4) any error or missing assumption, and (5) what follow-up question made the answer deeper. A fluent answer is not evidence that it is correct. Do not include raw call records, identifiers or student work in the prompt.

First, answer this claim on your own: **“A count from a randomly selected one of two equally long Poisson periods is Poisson with the average rate.”** Here, **selecting one** means: flip a fair coin; on heads choose period A and observe one count from A; on tails choose period B and observe one count from B. You observe one count, from one period. You do not add the two counts.

If $A\sim\operatorname{Poisson}(1)$ and $B\sim\operatorname{Poisson}(3)$, the selected variable $X$ has $P(X=k)=\tfrac12P(A=k)+\tfrac12P(B=k)$. This is a **mixture**: first choose which distribution supplies the observation, then draw from that distribution. It is different from $A+B$, where both counts are observed and added.

Consider the fair choice between period A and period B. Is the selected count Poisson(2)?

Ask an LLM to prove or refute the claim, showing its steps. Check each step yourself by conditioning on which period was selected, or with a short Python experiment. Your check should give $E[X]=2$ and $\operatorname{Var}(X)=3$ for the selected count, while a Poisson(2) count has variance 2. Explain why **selecting one** count gives a mixture, whereas **adding** independent Poisson counts gives a sum. Record any correction you made to the LLM's argument or to your own.


In [39]:
import random, statistics, math

rng = random.Random(2026)
n = 100_000

def draw_poisson(mu):
    return simulate_arrival_counts(1, 1, mu, rng)[0]

selected, summed = [], []
for _ in range(n):
    a, b = draw_poisson(1), draw_poisson(3)
    selected.append(a if rng.random() < 0.5 else b)   # on choisit UNE période
    summed.append(a + b)                               # on additionne les deux

for name, xs in (("sélection (mélange)", selected), ("somme A + B", summed)):
    print(f"{name:<20} moyenne {statistics.mean(xs):.2f}  variance {statistics.pvariance(xs):.2f}")
print("P(X=0) mélange :", round(selected.count(0) / n, 3), " Poisson(2) :", round(math.exp(-2), 3))

sélection (mélange)  moyenne 2.00  variance 3.01
somme A + B          moyenne 4.01  variance 4.04
P(X=0) mélange : 0.211  Poisson(2) : 0.135


## 5. Utiliser un LLM comme vérificateur de théorie

### Ma réponse avant de consulter l'LLM

**Claim.** « Un comptage tiré au hasard dans l'une de deux périodes de même durée, de Poisson, est Poisson de taux moyen. » Avec $A \sim \text{Poisson}(1)$, $B \sim \text{Poisson}(3)$, on lance une pièce équilibrée et on observe **un seul** comptage, celui de A ou de B. Le comptage observé $X$ est-il Poisson(2) ?

**Réponse : non.** $X$ est un **mélange** :

$$P(X = k) = \tfrac{1}{2} P(A = k) + \tfrac{1}{2} P(B = k)$$

- Espérance : $E[X] = \tfrac{1}{2} \cdot 1 + \tfrac{1}{2} \cdot 3 = 2$.
- Variance (formule de la variance totale) :

$$\text{Var}(X) = E[\text{Var}(X \mid \text{période})] + \text{Var}(E[X \mid \text{période}]) = 2 + 1 = 3$$

Une Poisson(2) a une variance de 2, donc $X$ n'est pas Poisson(2). On le voit aussi sur une probabilité : $P(X = 0) = \tfrac{1}{2}e^{-1} + \tfrac{1}{2}e^{-3} \approx 0.209$, contre $e^{-2} \approx 0.135$ pour une Poisson(2).

### Vérification par simulation (100 000 tirages, graine 2026)

| Quantité | Sélection (mélange) | Somme $A + B$ |
|---|---|---|
| Moyenne | ≈ 2.00 | ≈ 4.01 |
| Variance | ≈ 3.01 | ≈ 4.04 |

La sélection a bien une moyenne de 2 mais une variance proche de 3, donc elle n'est pas Poisson(2). La somme a une moyenne et une variance proches de 4 : c'est une Poisson(4).

### Sélection ou somme ?

- **Sélectionner** un comptage (pile ou face, puis on observe celui de A ou de B) donne un **mélange** : la moyenne est la moyenne des moyennes, mais la variance est plus grande, car elle contient aussi la variance entre les deux taux.
- **Additionner** deux comptages indépendants de Poisson donne une **somme** de Poisson, dont les taux s'additionnent ($1 + 3 = 4$).

**Lien avec les données.** C'est le mécanisme de la surdispersion observée sur novembre (variance/moyenne ≈ 14.8) : en regroupant tous les intervalles, on mélange des comptages venant de taux très différents (nuit, matinée, week-end).


## Wrap-up before the exit quiz

Write down: (1) one observation from the real counts, (2) one assumption of the first model, (3) one concrete model failure, and (4) one thing you still cannot infer about staffing. 


## Wrap-up

**1. Une observation sur les vrais comptages.**
Sur novembre, la moyenne est d'environ 14.1 entrées par intervalle de 15 min, mais 29.1 % des intervalles sont vides et la variance (≈ 209) est environ 15 fois la moyenne. La demande varie beaucoup selon l'heure : environ 0.1 entrée par intervalle à 03h contre 32 à 10h.

**2. Une hypothèse du premier modèle.**
Le taux d'arrivée est **constant** : un seul taux (≈ 0.943 entrée par minute) pour tous les intervalles du mois, quelle que soit l'heure ou le jour.

**3. Un échec concret du modèle.**
Sous le modèle, un intervalle vide a une probabilité d'environ $7 \times 10^{-7}$, soit 0.002 intervalle vide attendu sur 2 880, alors que j'en observe 837. Il surestime aussi la nuit (14.1 prédit contre 0.1 observé à 03h) et sous-estime le matin (14.5 contre 32.3 à 10h).

**4. Ce que je ne peux pas encore inférer sur le dimensionnement.**
Je ne peux pas dire si un agent de plus réduirait l'attente, car l'extrait ne contient ni les durées de service, ni les plannings des agents, ni les heures d'entrée et de sortie de file avec les abandons.